# 01 - Môi Trường, Cấu Hình Và Mô Hình Nền

Notebook này dùng để **chuẩn bị môi trường, endpoint và các mô hình embedding nền**. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.

### BƯỚC 1: Kiểm Tra Môi Trường

- **Tác dụng chính:** Notebook này dùng để chuẩn bị môi trường, endpoint và các mô hình embedding nền. Phần triển khai dài được thu gọn mặc định; các cell thực thi vẫn giữ output cần thiết cho debug.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** các lệnh trong bước cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [13]:
import sys


In [14]:
print(f"Python executable: {sys.executable}")

print(f"Python version   : {sys.version}")

print("\nGợi ý kiểm tra:")

print("- Nếu dùng venv, đường dẫn nên chứa thư mục venv của project.")

print("- Nếu dùng conda, đường dẫn nên thuộc env bạn đã cài torch/langchain/qdrant.")


Python executable: c:\Users\DELL\anaconda3\python.exe
Python version   : 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]

Gợi ý kiểm tra:
- Nếu dùng venv, đường dẫn nên chứa thư mục venv của project.
- Nếu dùng conda, đường dẫn nên thuộc env bạn đã cài torch/langchain/qdrant.


### BƯỚC 2: Import Thư Viện Cần Cho Notebook Này

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** các lệnh trong bước cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [15]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings
from PIL import Image
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor

In [16]:
print("[OK] Import tối thiểu cho notebook 01 đã hoàn tất")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device     : {torch.cuda.get_device_name(0)}")
else:
    print("CUDA device     : CPU only")

[OK] Import tối thiểu cho notebook 01 đã hoàn tất
PyTorch version : 2.7.0+cpu
CUDA available  : False
CUDA device     : CPU only


### BƯỚC 3: Constants Và Cấu Hình Mô Hình

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `OLLAMA_BASE_URL`, `QDRANT_URL`, `VIFASHIONCLIP_SERVICE_URL`, `LLM_MODEL`, `QWEN_VL_MODEL` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
# ════════════════════════════════════════════════════════════════
# Service endpoints
# ════════════════════════════════════════════════════════════════
OLLAMA_BASE_URL = "http://localhost:11434"

QDRANT_URL = "http://localhost:6333"

VIFASHIONCLIP_SERVICE_URL = "http://localhost:18080"

# ════════════════════════════════════════════════════════════════
# LLM & Vision-Language models chạy local qua Ollama
# ════════════════════════════════════════════════════════════════

LLM_MODEL = "qwen3:4b-instruct"       # Chat chính, router fallback, summarize

QWEN_VL_MODEL = "qwen2.5vl:3b"        # Phân tích ảnh người/sản phẩm

# ════════════════════════════════════════════════════════════════
# ViFashionCLIP model names dùng cho Layer A text retrieval
# Teacher tạo không gian FashionCLIP 512 chiều.
# Student tiếng Việt học cách đi vào không gian đó.
# ════════════════════════════════════════════════════════════════

TEACHER_MODEL_NAME = "patrickjohncyh/fashion-clip"

STUDENT_MODEL_NAME = "AITeamVN/Vietnamese_Embedding_v2"

STUDENT_MAX_LENGTH = 128

# ════════════════════════════════════════════════════════════════
# ProjectionHead: map student_hidden_dim -> FashionCLIP 512-dim
# ════════════════════════════════════════════════════════════════

PROJECTION_HIDDEN_DIM = 1024

PROJECTION_NUM_LAYERS = 3

PROJECTION_DROPOUT = 0.05

# ════════════════════════════════════════════════════════════════
# Qdrant collections và kích thước vector kỳ vọng
# ════════════════════════════════════════════════════════════════

PRODUCT_COLLECTION = "fashion_products_vifashionclip_vi_65k_structured_vi"

PRODUCT_IMAGE_COLLECTION = "fashion_products_fashionclip_image_main_65k"

LAYER_B_FEMALE_COLLECTION = "layer_b_female"

LAYER_B_MALE_COLLECTION = "layer_b_male"

PRODUCT_VECTOR_SIZE = 512

IMAGE_VECTOR_SIZE = 512

LAYER_B_VECTOR_SIZE = 1024

# ════════════════════════════════════════════════════════════════
# Retrieval knobs: các số này ảnh hưởng trực tiếp đến tốc độ và chất lượng
# ════════════════════════════════════════════════════════════════

PRODUCT_SEARCH_CANDIDATE_K = 15  # Qdrant lấy raw candidates

RERANKER_TOP_N = 8               # Reranker chấm lại top-N candidates

PRODUCT_SEARCH_PAGE_SIZE = 5      # Số sản phẩm tối đa gửi vào LLM

PRODUCT_SEARCH_BRAND_LIMIT = 2    # Giảm lặp quá nhiều sản phẩm cùng brand


In [3]:
print("[OK] Cấu hình chính")

print(f"LLM                 : {LLM_MODEL}")

print(f"Vision              : {QWEN_VL_MODEL}")

print(f"Layer A text model  : {STUDENT_MODEL_NAME} -> FashionCLIP 512-dim")

print(f"Layer A text store  : {PRODUCT_COLLECTION} ({PRODUCT_VECTOR_SIZE}-dim)")

print(f"Layer A image store : {PRODUCT_IMAGE_COLLECTION} ({IMAGE_VECTOR_SIZE}-dim)")

print(f"Layer B stores      : {LAYER_B_FEMALE_COLLECTION}, {LAYER_B_MALE_COLLECTION} ({LAYER_B_VECTOR_SIZE}-dim)")

print(f"Retrieval           : top-{PRODUCT_SEARCH_CANDIDATE_K} -> rerank top-{RERANKER_TOP_N} -> final {PRODUCT_SEARCH_PAGE_SIZE}")


[OK] Cấu hình chính
LLM                 : qwen3:4b-instruct
Vision              : qwen2.5vl:3b
Layer A text model  : AITeamVN/Vietnamese_Embedding_v2 -> FashionCLIP 512-dim
Layer A text store  : fashion_products_vifashionclip_vi_65k_structured_vi (512-dim)
Layer A image store : fashion_products_fashionclip_image_main_65k (512-dim)
Layer B stores      : layer_b_female, layer_b_male (1024-dim)
Retrieval           : top-15 -> rerank top-8 -> final 5


### BƯỚC 4: Tìm Thư Mục Dự Án Và Checkpoint

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `resolve_chatbot_fashion_dir`, `CHATBOT_FASHION_DIR`, `VIFASHIONCLIP_CHECKPOINT` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
def resolve_chatbot_fashion_dir() -> Path:
    """Tìm thư mục gốc Chatbot_Fashion/ từ vị trí hiện tại.

    Args:
        Không có.

    Returns:
        Path: Dữ liệu đã được chuẩn hóa hoặc suy ra để dùng ở bước tiếp theo.
    """
    cwd = Path.cwd()

    if cwd.name == "notebooks" and cwd.parent.name == "Chatbot_Fashion":
        return cwd.parent
    if cwd.name == "Chatbot_Fashion":
        return cwd

    candidate = cwd / "Chatbot_Fashion"
    if candidate.exists():
        return candidate

    for parent in cwd.parents:
        candidate = parent / "Chatbot_Fashion"
        if candidate.exists():
            return candidate

    raise FileNotFoundError("Không tìm thấy Chatbot_Fashion/ từ thư mục hiện tại")


In [4]:
CHATBOT_FASHION_DIR = resolve_chatbot_fashion_dir()

VIFASHIONCLIP_CHECKPOINT = (
    CHATBOT_FASHION_DIR
    / "Vietnamese"
    / "vifashionclip_aiteamvn_embedding_v2_projection_336k"
    / "stage2_last_layers"
    / "best_stage2_model.pt"
)

print(f"[OK] Thư mục dự án      : {CHATBOT_FASHION_DIR}")

print(f"[OK] Checkpoint path    : {VIFASHIONCLIP_CHECKPOINT}")

print(f"     Checkpoint exists  : {VIFASHIONCLIP_CHECKPOINT.exists()}")

if not VIFASHIONCLIP_CHECKPOINT.exists():
    print("[CẢNH BÁO] Chưa thấy checkpoint. Layer A text embedding sẽ không load được.")


[OK] Thư mục dự án      : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion
[OK] Checkpoint path    : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\Vietnamese\vifashionclip_aiteamvn_embedding_v2_projection_336k\stage2_last_layers\best_stage2_model.pt
     Checkpoint exists  : True


### BƯỚC 5: Kiểm Tra Nhanh Các Dịch Vụ Nền

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `ping_http`, `service_status` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
from urllib.error import URLError

from urllib.request import urlopen

def ping_http(name: str, url: str, timeout: float = 2.0) -> bool:
    """Ping một HTTP endpoint và in kết quả gọn, không làm notebook crash nếu service tắt.

    Args:
        name (str): Giá trị đầu vào phục vụ bước xử lý này.
        url (str): Giá trị đầu vào phục vụ bước xử lý này.
        timeout (float): Thời gian chờ tối đa cho thao tác bên ngoài.

    Returns:
        bool: Kết quả đã xử lý của hàm.
    """
    try:
        with urlopen(url, timeout=timeout) as response:
            status = getattr(response, "status", "unknown")
        print(f"[OK] {name:<24} {url} -> HTTP {status}")
        return True
    except URLError as exc:
        print(f"[FAIL] {name:<22} {url} -> {exc}")
        return False
    except Exception as exc:
        print(f"[FAIL] {name:<22} {url} -> {type(exc).__name__}: {exc}")
        return False


In [17]:
service_status = {
    "qdrant": ping_http("Qdrant", f"{QDRANT_URL}/collections"),
    "ollama": ping_http("Ollama", f"{OLLAMA_BASE_URL}/api/tags"),
    "vifashionclip_service": ping_http("ViFashionCLIP service", f"{VIFASHIONCLIP_SERVICE_URL}/health"),
}

print("\nTóm tắt service:", service_status)


[OK] Qdrant                   http://localhost:6333/collections -> HTTP 200
[OK] Ollama                   http://localhost:11434/api/tags -> HTTP 200
[OK] ViFashionCLIP service    http://localhost:18080/health -> HTTP 200

Tóm tắt service: {'qdrant': True, 'ollama': True, 'vifashionclip_service': True}


### BƯỚC 6: Các Khối Neural Network Của ViFashionCLIP

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `ResidualMLPBlock` để các bước sau tiếp tục sử dụng.


In [6]:
class ResidualMLPBlock(nn.Module):
    """
    Một MLP block có residual connection.

    Input:
        x: tensor shape [batch_size, dim]

    Output:
        tensor shape [batch_size, dim]

    Vì sao cần residual?
        ProjectionHead phải học cách biến embedding tiếng Việt sang không gian FashionCLIP.
        Nếu chỉ xếp nhiều Linear layer, tín hiệu gốc có thể bị biến dạng mạnh.
        Công thức x + block(x) cho phép model giữ lại phần hữu ích từ input,
        đồng thời học thêm phần hiệu chỉnh cần thiết.
    """

    def __init__(self, dim: int, dropout: float = 0.05):
        """Xử lý bước `init` của pipeline.

        Args:
            dim (int): Giá trị đầu vào phục vụ bước xử lý này.
            dropout (float): Giá trị đầu vào phục vụ bước xử lý này.

        Returns:
            None: Không trả về; khởi tạo trạng thái cho đối tượng.
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),   # Mở rộng chiều để học tương tác phong phú hơn
            nn.GELU(),                 # Activation mượt, thường dùng trong transformer
            nn.Dropout(dropout),       # Giảm overfit khi training
            nn.Linear(dim * 2, dim),   # Thu về đúng chiều ban đầu để cộng residual
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


### Bước 11: Chuẩn bị môi trường, endpoint và các mô hình embedding nền

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `ProjectionHead` để các bước sau tiếp tục sử dụng.


In [7]:
class ProjectionHead(nn.Module):
    """
    Ánh xạ embedding của student encoder sang không gian FashionCLIP 512 chiều.

    Tham số quan trọng:
        input_dim: hidden size của Vietnamese student encoder.
        output_dim: thường là 512, lấy từ checkpoint teacher_dim.
        hidden_dim: chiều trung gian của MLP.
        num_layers: số lớp biến đổi. Với 3 layer, sẽ có 1 ResidualMLPBlock ở giữa.

    Góc nhìn retrieval:
        Qdrant không hiểu text, nó chỉ so sánh vector.
        ProjectionHead quyết định vector tiếng Việt có nằm đúng “tọa độ thời trang” hay không.
    """

    def __init__(self, input_dim, output_dim, hidden_dim=1024, num_layers=3, dropout=0.05):
        """Xử lý bước `init` của pipeline.

        Args:
            input_dim (Any): Giá trị đầu vào phục vụ bước xử lý này.
            output_dim (Any): Giá trị đầu vào phục vụ bước xử lý này.
            hidden_dim (Any): Giá trị đầu vào phục vụ bước xử lý này.
            num_layers (Any): Giá trị đầu vào phục vụ bước xử lý này.
            dropout (Any): Giá trị đầu vào phục vụ bước xử lý này.

        Returns:
            None: Không trả về; khởi tạo trạng thái cho đối tượng.
        """
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ]

        for _ in range(max(0, num_layers - 2)):
            layers.append(ResidualMLPBlock(hidden_dim, dropout=dropout))

        layers.extend([
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),  # Output kỳ vọng: 512 chiều
        ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


### Bước 12: Chuẩn bị môi trường, endpoint và các mô hình embedding nền

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `mean_pool` để các bước sau tiếp tục sử dụng.


In [8]:
def mean_pool(last_hidden_state, attention_mask):
    """Gom token embeddings thành một sentence embedding.

    Args:
        last_hidden_state (Any): Dữ liệu trạng thái hoặc ngữ cảnh có cấu trúc.
        attention_mask (Any): Giá trị đầu vào phục vụ bước xử lý này.

    Returns:
        Any: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


### Bước 13: Chuẩn bị môi trường, endpoint và các mô hình embedding nền

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `Stage2StudentProjection` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
class Stage2StudentProjection(nn.Module):
    """
    Model inference hoàn chỉnh của ViFashionCLIP text.

    Luồng forward:
        input_ids + attention_mask
          -> student encoder
          -> mean_pool
          -> projection_head
          -> vector 512 chiều

    Tên "Stage2" đến từ quá trình train: student encoder + projection head
    đã được tinh chỉnh để tiệm cận không gian FashionCLIP teacher.
    """

    def __init__(self, encoder, projection_head):
        """Xử lý bước `init` của pipeline.

        Args:
            encoder (Any): Giá trị đầu vào phục vụ bước xử lý này.
            projection_head (Any): Giá trị đầu vào phục vụ bước xử lý này.

        Returns:
            None: Không trả về; khởi tạo trạng thái cho đối tượng.
        """
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, input_ids, attention_mask):
        """Xử lý bước `forward` của pipeline.

        Args:
            input_ids (Any): Giá trị đầu vào phục vụ bước xử lý này.
            attention_mask (Any): Giá trị đầu vào phục vụ bước xử lý này.

        Returns:
            Any: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
        """
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        return self.projection_head(pooled)


In [9]:
print("[OK] Neural blocks đã sẵn sàng: ResidualMLPBlock | ProjectionHead | mean_pool | Stage2StudentProjection")


[OK] Neural blocks đã sẵn sàng: ResidualMLPBlock | ProjectionHead | mean_pool | Stage2StudentProjection


### BƯỚC 7: BGEM3Embeddings Cho Layer B

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `BGEM3Embeddings` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
class BGEM3Embeddings(Embeddings):
    """
    LangChain wrapper cho BGE-M3 qua Ollama.

    Dùng khi:
        - Index rule phối đồ vào Layer B.
        - Embed query kiểu "phối đồ đi tiệc cho dáng quả lê" để tìm rule phù hợp.

    Không dùng khi:
        - Tìm sản phẩm thật trong kho. Việc đó thuộc Layer A và dùng ViFashionCLIP.
    """

    def __init__(self, model_name: str = "bge-m3"):
        # OllamaEmbeddings không load model trong Python process này.
        # Nó gửi request sang Ollama server ở localhost:11434.
        self.ollama_embeddings = OllamaEmbeddings(
            model=model_name,
            base_url=OLLAMA_BASE_URL,
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều rule stylist khi index Layer B vào Qdrant."""
        return self.ollama_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed một query phối đồ khi cần tìm rule phù hợp."""
        return self.ollama_embeddings.embed_query(text)


In [10]:
print("[OK] BGEM3Embeddings đã được định nghĩa cho Layer B (1024-dim)")


[OK] BGEM3Embeddings đã được định nghĩa cho Layer B (1024-dim)


### BƯỚC 8: ViFashionCLIPTextEmbeddings Cho Layer A Text

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `ViFashionCLIPTextEmbeddings` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
class ViFashionCLIPTextEmbeddings(Embeddings):
    """
    LangChain Embeddings wrapper cho ViFashionCLIP tiếng Việt.

    Dùng cho Layer A text:
        - Embed 65k sản phẩm khi index.
        - Embed query tiếng Việt khi user tìm sản phẩm.

    Output:
        vector 512 chiều, cùng không gian với FashionCLIP teacher.

    Lưu ý tốc độ:
        Khởi tạo class này có thể chậm vì phải load tokenizer, encoder và checkpoint.
        Trong app thật nên lazy-load hoặc dùng remote embedding service.
    """

    def __init__(
        self,
        checkpoint_path: str | Path = VIFASHIONCLIP_CHECKPOINT,
        batch_size: int = 32,
    ):
        """Xử lý bước `init` của pipeline.

        Args:
            checkpoint_path (str | Path): Đường dẫn dữ liệu hoặc tệp cần xử lý.
            batch_size (int): Giới hạn số phần tử được xử lý hoặc trả về.

        Returns:
            None: Không trả về; khởi tạo trạng thái cho đối tượng.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.checkpoint_path = Path(checkpoint_path)

        if not self.checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint không tồn tại: {self.checkpoint_path}")

        # Checkpoint là nguồn sự thật: nó cho biết student model và teacher_dim đã dùng khi train.
        checkpoint = torch.load(self.checkpoint_path, map_location="cpu")
        student_name = checkpoint.get("student_model_name", STUDENT_MODEL_NAME)
        teacher_dim = int(checkpoint.get("teacher_dim", 512))

        # Tokenizer biến text thành input_ids/attention_mask.
        self.tokenizer = AutoTokenizer.from_pretrained(student_name)

        # Encoder tạo token embeddings tiếng Việt.
        encoder = AutoModel.from_pretrained(student_name)

        # ProjectionHead đưa embedding tiếng Việt sang không gian FashionCLIP.
        projection = ProjectionHead(
            input_dim=encoder.config.hidden_size,
            output_dim=teacher_dim,
            hidden_dim=PROJECTION_HIDDEN_DIM,
            num_layers=PROJECTION_NUM_LAYERS,
            dropout=PROJECTION_DROPOUT,
        )

        # Load đúng trọng số đã train cho cả encoder và projection.
        encoder.load_state_dict(checkpoint["encoder_state_dict"])
        projection.load_state_dict(checkpoint["projection_state_dict"])

        # eval() + requires_grad=False: chỉ inference, không train trong chatbot.
        self.model = Stage2StudentProjection(encoder, projection).to(self.device).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.embedding_dim = teacher_dim
        print(f"[OK] ViFashionCLIP loaded: {self.checkpoint_path}")
        print(f"     Student: {student_name}")
        print(f"     Device : {self.device} | Dim: {self.embedding_dim}")

    @torch.no_grad()
    def _encode(self, texts: list[str]) -> list[list[float]]:
        """Encode một batch text thành vector 512 chiều.

        Args:
            texts (list[str]): Giá trị đầu vào phục vụ bước xử lý này.

        Returns:
            list[list[float]]: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
        """
        all_embeds = []
        for start in tqdm(range(0, len(texts), self.batch_size), desc="ViFashionCLIP embed", leave=False):
            batch = [str(text) for text in texts[start:start + self.batch_size]]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=STUDENT_MAX_LENGTH,
                return_tensors="pt",
            ).to(self.device)

            embeds = self.model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
            )
            # Chuẩn hóa vector về cùng thang đo để phép so sánh cosine ổn định.
            embeds = F.normalize(embeds, p=2, dim=-1)
            all_embeds.append(embeds.detach().cpu())

        return torch.cat(all_embeds, dim=0).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều product documents khi index Layer A vào Qdrant."""
        return self._encode(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed một query tiếng Việt của người dùng khi search sản phẩm."""
        return self._encode([text])[0]


In [11]:
print("[OK] ViFashionCLIPTextEmbeddings đã được định nghĩa cho Layer A text (512-dim)")


[OK] ViFashionCLIPTextEmbeddings đã được định nghĩa cho Layer A text (512-dim)


### BƯỚC 9: FashionCLIPImageEmbeddings Cho Layer A Image

- **Tác dụng chính:** Thực hiện bước chuẩn bị môi trường, endpoint và các mô hình embedding nền.
- **Đầu vào (Input):** Cấu hình, dữ liệu hoặc đối tượng đã được chuẩn bị ở bước trước; tham số chi tiết nằm trong docstring của hàm.
- **Đầu ra (Output):** `FashionCLIPImageEmbeddings` cùng output debug như shape, ID, score hoặc trạng thái trung gian.


In [ ]:
class FashionCLIPImageEmbeddings:
    """
    FashionCLIP image encoder cho image retrieval.

    Dùng khi:
        - Index ảnh MAIN của sản phẩm vào Qdrant.
        - Encode ảnh user gửi để tìm sản phẩm tương tự.

    Không dùng khi:
        - Tìm rule phối đồ Layer B.
        - Embed query text tiếng Việt.
    """

    def __init__(self, model_name: str = TEACHER_MODEL_NAME, batch_size: int = 32):
        """Xử lý bước `init` của pipeline.

        Args:
            model_name (str): Giá trị đầu vào phục vụ bước xử lý này.
            batch_size (int): Giới hạn số phần tử được xử lý hoặc trả về.

        Returns:
            None: Không trả về; khởi tạo trạng thái cho đối tượng.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.model_name = model_name

        # Processor chuẩn hóa ảnh đúng format mà FashionCLIP cần.
        self.processor = CLIPProcessor.from_pretrained(model_name)

        # Model tạo image features trong cùng không gian 512 chiều của FashionCLIP.
        self.model = CLIPModel.from_pretrained(model_name).to(self.device).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.embedding_dim = int(getattr(self.model.config, "projection_dim", 512))
        print(f"[OK] FashionCLIP image encoder: {model_name}")
        print(f"     Device: {self.device} | Dim: {self.embedding_dim}")

    @staticmethod
    def _open_image(image_path: str | Path):
        """Mở ảnh và convert sang RGB để xử lý PNG/JPEG nhất quán."""
        return Image.open(image_path).convert("RGB")

    @torch.no_grad()
    def encode_image_paths(self, image_paths: list[str | Path]) -> list[list[float] | None]:
        """Encode nhiều ảnh thành vector 512 chiều.

        Args:
            image_paths (list[str | Path]): Ảnh hoặc thông tin ảnh đầu vào.

        Returns:
            list[list[float] | None]: Vector hoặc tensor biểu diễn dữ liệu đầu vào.
        """
        results: list[list[float] | None] = [None] * len(image_paths)

        for start in tqdm(range(0, len(image_paths), self.batch_size), desc="FashionCLIP image embed", leave=False):
            batch_paths = image_paths[start:start + self.batch_size]
            images = []
            valid_positions = []

            for offset, path in enumerate(batch_paths):
                try:
                    images.append(self._open_image(path))
                    valid_positions.append(start + offset)
                except Exception as exc:
                    print(f"[WARN] Không đọc được ảnh {path}: {exc}")

            if not images:
                continue

            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device) # type: ignore
            embeds = self.model.get_image_features(**inputs)
            if hasattr(embeds, "pooler_output"):
                embeds = embeds.pooler_output
            # Chuẩn hóa vector về cùng thang đo để phép so sánh cosine ổn định.
            embeds = F.normalize(embeds, p=2, dim=-1).detach().cpu().tolist()

            for position, vector in zip(valid_positions, embeds):
                results[position] = vector

        return results

    def embed_image(self, image_path: str | Path) -> list[float] | None:
        """Encode một ảnh query của user để search sản phẩm tương tự."""
        return self.encode_image_paths([image_path])[0]


In [12]:
print("[OK] FashionCLIPImageEmbeddings đã được định nghĩa cho Image Retrieval (512-dim)")

print("     Lưu ý: chỉ khởi tạo class này khi cần xử lý ảnh để tránh tốn RAM/GPU.")


[OK] FashionCLIPImageEmbeddings đã được định nghĩa cho Image Retrieval (512-dim)
     Lưu ý: chỉ khởi tạo class này khi cần xử lý ảnh để tránh tốn RAM/GPU.
